<a href="https://colab.research.google.com/github/Rogerkamana/assignment2-rkv1/blob/main/Assignment_3_Python_Refresher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 3 — Python Refresher
## Library Management System


In [1]:
# SETUP — Required libraries
import pickle
import sqlite3
import requests
from pathlib import Path
print("Required libraries imported successfully.")

Required libraries imported successfully.


### 1. Data Types and Variables

In [2]:
# 1. DATA TYPES AND VARIABLES
# Run this cell after the preceding sections.

book_title = "Python Programming"
book_quantity = 5
book_price = 29.99
is_available = True
print(f"Book: {book_title}, Quantity: {book_quantity}, Price: ${book_price:.2f}, Available: {is_available}")

Book: Python Programming, Quantity: 5, Price: $29.99, Available: True


### 2. Lists

In [3]:
# 2. LISTS — Add books (TODO 1 of 6)
# Run this cell after the preceding sections.

books = ["Python Basics", "Advanced Python", "Data Science with Python"]
books.append("Machine Learning with Python")
books.insert(1, "Introduction to AI")
print("Updated book list:", books)

Updated book list: ['Python Basics', 'Introduction to AI', 'Advanced Python', 'Data Science with Python', 'Machine Learning with Python']


### 3. Loops and range()

In [4]:
# 3. LOOPS — Print books with indices (TODO 2 of 6)
# Run this cell after the preceding sections.

print("Available books:")
for i in range(len(books)):
    print(f"{i + 1}. {books[i]}")

Available books:
1. Python Basics
2. Introduction to AI
3. Advanced Python
4. Data Science with Python
5. Machine Learning with Python


### 4. User Input
Enter a title when prompted. Empty titles are ignored. In an automated run, the default is used if input is unavailable.

In [9]:
# 4. USER INPUT
# Run this cell after the preceding sections.

try:
    new_book = input("Enter a new book title (press Enter to skip): ").strip()
except (EOFError, KeyboardInterrupt):
    new_book = "Introduction to povery measurement"
if new_book:
    books.append(new_book)
    print(f"Added: {new_book}")
else:
    print("No title entered; library unchanged.")
print("Updated book list:", books)

Enter a new book title (press Enter to skip): Introduction to povery measurement
Added: Introduction to povery measurement
Updated book list: ['Python Basics', 'Introduction to AI', 'Advanced Python', 'Data Science with Python', 'Machine Learning with Python', 'Poverty', 'Introduction to povery measurement']


### 5. Functions and Conditional Statements

In [10]:
# 5. FUNCTIONS AND CONDITIONAL STATEMENTS — Check availability (TODO 3 of 6)
# Run this cell after the preceding sections.

def check_availability(book_name):
    """Return True if a nonempty book title exists in the library (case-insensitive)."""
    if not isinstance(book_name, str) or not book_name.strip():
        return False
    return any(book.casefold() == book_name.strip().casefold() for book in books)

print("Is Python Basics available?", check_availability("Python Basics"))
print("Is Unknown Book available?", check_availability("Unknown Book"))
assert check_availability("python basics") is True
assert check_availability("Unknown Book") is False

Is Python Basics available? True
Is Unknown Book available? False


### 6. File I/O

In [11]:
# 6. FILE I/O — Read books from a file (TODO 4 of 6)
# Run this cell after the preceding sections.

def save_books_to_file(filename):
    """Write one book title per line to a UTF-8 text file."""
    try:
        with open(filename, "w", encoding="utf-8") as f:
            for book in books:
                f.write(f"{book}\n")
        return True
    except OSError as exc:
        print(f"Could not save books: {exc}")
        return False

def read_books_from_file(filename):
    """Return nonempty titles from a text file, or [] on file errors."""
    try:
        with open(filename, "r", encoding="utf-8") as f:
            return [line.strip() for line in f if line.strip()]
    except (OSError, UnicodeError) as exc:
        print(f"Could not read books: {exc}")
        return []

if save_books_to_file("books.txt"):
    loaded_text_books = read_books_from_file("books.txt")
    print("Books read from file:", loaded_text_books)
    assert loaded_text_books == books
print("Missing file test:", read_books_from_file("missing_books.txt"))

Books read from file: ['Python Basics', 'Introduction to AI', 'Advanced Python', 'Data Science with Python', 'Machine Learning with Python', 'Poverty', 'Introduction to povery measurement']
Could not read books: [Errno 2] No such file or directory: 'missing_books.txt'
Missing file test: []


### 7. Pickle for Serialization
Only unpickle files you trust. Pickle files can execute malicious code when loaded.

In [12]:
# 7. PICKLE SERIALIZATION — Load books (TODO 5 of 6)
# Run this cell after the preceding sections.

def save_books_pickle(filename):
    try:
        with open(filename, "wb") as f:
            pickle.dump(books, f)
        return True
    except (OSError, pickle.PickleError) as exc:
        print(f"Could not save pickle: {exc}")
        return False

def load_books_pickle(filename):
    """Load a trusted pickle file and verify it contains a list of titles."""
    try:
        with open(filename, "rb") as f:
            result = pickle.load(f)
        if not isinstance(result, list) or not all(isinstance(x, str) for x in result):
            print("Invalid pickle content: expected a list of book titles.")
            return []
        return result
    except (OSError, pickle.PickleError, EOFError, AttributeError, ImportError) as exc:
        print(f"Could not load pickle: {exc}")
        return []

if save_books_pickle("books.pkl"):
    loaded_pickle_books = load_books_pickle("books.pkl")
    print("Books loaded from pickle:", loaded_pickle_books)
    assert loaded_pickle_books == books

Books loaded from pickle: ['Python Basics', 'Introduction to AI', 'Advanced Python', 'Data Science with Python', 'Machine Learning with Python', 'Poverty', 'Introduction to povery measurement']


### 8. SQLite Database Interaction

In [13]:
# 8. SQLITE DATABASE — Add books (TODO 6 of 6)
# Run this cell after the preceding sections.

DB_FILE = "library.db"

def create_books_table():
    try:
        with sqlite3.connect(DB_FILE) as conn:
            conn.execute("""CREATE TABLE IF NOT EXISTS books
                            (id INTEGER PRIMARY KEY, title TEXT, quantity INTEGER)""")
        print("Books table ready.")
        return True
    except sqlite3.Error as exc:
        print(f"Database setup error: {exc}")
        return False

def add_book_to_db(title, quantity):
    """Insert a book using a parameterized query; return True on success."""
    if not isinstance(title, str) or not title.strip():
        print("Book title must not be empty.")
        return False
    if isinstance(quantity, bool) or not isinstance(quantity, int) or quantity < 0:
        print("Quantity must be a nonnegative integer.")
        return False
    try:
        with sqlite3.connect(DB_FILE) as conn:
            conn.execute("INSERT INTO books (title, quantity) VALUES (?, ?)",
                         (title.strip(), quantity))
        print(f"Added to database: {title.strip()} (quantity: {quantity})")
        return True
    except sqlite3.Error as exc:
        print(f"Database insert error: {exc}")
        return False

def show_database_books():
    try:
        with sqlite3.connect(DB_FILE) as conn:
            rows = conn.execute("SELECT id, title, quantity FROM books ORDER BY id").fetchall()
        print("Database books (ID, title, quantity):")
        for row in rows:
            print(row)
        return rows
    except sqlite3.Error as exc:
        print(f"Database read error: {exc}")
        return []

if create_books_table():
    assert add_book_to_db("Python for Beginners", 10)
    assert not add_book_to_db("", -1)
    show_database_books()

Books table ready.
Added to database: Python for Beginners (quantity: 10)
Book title must not be empty.
Database books (ID, title, quantity):
(1, 'Python for Beginners', 10)


### 9. Web API Interaction (GitHub API)
The provided GitHub API example is preserved, with timeout, status checking, and graceful handling of network or response errors. Run this cell in Colab with internet access.

In [14]:
# 9. WEB API — GitHub repositories
# Run this cell after the preceding sections.

def get_python_repos():
    url = "https://api.github.com/search/repositories"
    params = {"q": "language:python", "sort": "stars", "order": "desc"}
    try:
        response = requests.get(url, params=params, timeout=15)
        response.raise_for_status()
        data = response.json()
        return [repo["name"] for repo in data.get("items", [])[:5] if "name" in repo]
    except (requests.RequestException, ValueError, KeyError, TypeError) as exc:
        print(f"GitHub API request failed: {exc}")
        return []

print("Top 5 Python repositories on GitHub:", get_python_repos())
print("\nLibrary Management System operations completed.")

GitHub API request failed: 403 Client Error: rate limit exceeded for url: https://api.github.com/search/repositories?q=language%3Apython&sort=stars&order=desc
Top 5 Python repositories on GitHub: []

Library Management System operations completed.
